## E8

**Query:**

```sql
SELECT filmid FROM F
WHERE NOT EXISTS (
    SELECT * FROM S, A
    WHERE geslacht = 'M' AND S.filmid = F.filmid
)
```

**Uitleg:**

Voor elke film wordt gecontroleerd of er **geen tuple bestaat** waarbij:

* het geslacht 'M' is
* en de film overeenkomt

Probleem: er ontbreekt een koppeling tussen $S$ en $A$ (geen $S.actid = A.actid$).
Daardoor ontstaat een cartesiaans product.

$\Rightarrow$ Logisch bedoeld: films zonder mannelijke acteurs
$\Rightarrow$ Maar: fout door ontbrekende join

---

## E9

**Query:**

```sql
SELECT filmid FROM F
WHERE filmid NOT IN (
    SELECT filmid FROM S, A
    WHERE geslacht = 'M' AND S.actid = A.actid
)
```

**Uitleg:**

De binnenste query geeft alle films met een mannelijke acteur.
De buitenste query selecteert films die daar **niet in voorkomen**.

$\Rightarrow$ Resultaat: films waarin geen enkele man speelt

---

## E10

**Query:**

```sql
SELECT F1.titel FROM F F1, F F2
WHERE F1.titel = F2.titel
AND F1.jaar = F2.jaar
AND F1.filmid <> F2.filmid
```

**Uitleg:**

Self-join op $F$:

* zelfde titel
* zelfde jaar
* verschillende filmid

$\Rightarrow$ Resultaat: titels die meerdere keren voorkomen in hetzelfde jaar

---

## E11

**Query:**

```sql
SELECT naam FROM A
WHERE NOT EXISTS (
    SELECT * FROM F
    WHERE regisseur = 'Scorsese'
    AND jaar >= 1990 AND jaar <= 1991
    AND NOT EXISTS (
        SELECT * FROM S
        WHERE S.filmid = F.filmid
        AND S.actid = A.actid
))
```

**Uitleg:**

Structuur: dubbele NOT EXISTS.

Binnenste:

* er bestaat geen record → acteur zit niet in film

Buitenste:

* er bestaat geen film waarvoor dat geldt

$\Rightarrow$ Resultaat: acteur zit in alle geselecteerde films

---

## E12

**Query:**

```sql
SELECT naam FROM A
WHERE NOT EXISTS (
    SELECT * FROM S
    WHERE S.actid = A.actid
    AND NOT EXISTS (
        SELECT * FROM F
        WHERE regisseur = 'Scorsese'
        AND jaar >= 1990 AND jaar <= 1991
        WHERE S.filmid = F.filmid
        AND S.actid = A.actid
))
```

**Uitleg:**

Problemen:

* dubbele WHERE
* syntactisch ongeldig
* logica incorrect

$\Rightarrow$ Query werkt niet

---

## E13

**Query:**

```sql
SELECT actid FROM S
GROUP BY actid
HAVING MAX(COUNT(*))
```

**Uitleg:**

Probleem:

* $COUNT(*)$ is al aggregatie
* $MAX(COUNT(*))$ is niet toegestaan

$\Rightarrow$ Ongeldig

---

## E14

**Query:**

```sql
SELECT actid FROM S
GROUP BY actid
HAVING SELECT MAX(COUNT(*))
FROM S
GROUP BY actid
)
```

**Uitleg:**

Problemen:

* fout gebruik van HAVING
* syntax incorrect
* haakjes kloppen niet

$\Rightarrow$ Ongeldig

---

## E15

**Query:**

```sql
SELECT actid FROM S
GROUP BY actid
HAVING COUNT(*) >= ALL (
    SELECT SUM(*) FROM S
    GROUP BY actid
)
```

**Uitleg:**

Probleem:

* $SUM(*)$ bestaat niet

$\Rightarrow$ Ongeldig

---

## E16

**Query:**

```sql
SELECT actid FROM S
GROUP BY actid
HAVING COUNT(*) >= ALL (
    SELECT COUNT(*) FROM S
    GROUP BY actid
)
```

**Uitleg:**

* tel aantal films per acteur
* vergelijk met alle andere aantallen
* selecteer maximale waarde

$\Rightarrow$ Resultaat: acteur(s) met het hoogste aantal films

---

## Samenvattende patronen

* $NOT \ EXISTS$ $\Rightarrow$ “er bestaat geen …”
* $NOT \ EXISTS ( \dots NOT \ EXISTS \dots )$ $\Rightarrow$ “voor alle …”
* $GROUP \ BY + COUNT$ $\Rightarrow$ tellen per groep
* $>= ALL$ $\Rightarrow$ maximum selecteren
